# Structural and Functional Data Preparation

This notebook produces the input files required by the H1 cross-check and scope checks on the 93-neuron scan-9.3 cohort.

**Files produced:**

| File | Used for |
|------|-----------|
| `data/G_93.pkl` | Intermediate graph (loaded by Part 2) |
| `data/exports/G_93_nodes.csv` | 93-neuron node attributes (structural) |
| `data/exports/G_93_edges.csv` | 93-neuron directed synapses |
| `outputs/functional_network/F_correlation_matrix.npy` | 93×93 trace-correlation matrix |
| `outputs/functional_network/functional_cohort.csv` | 93-neuron cohort metadata |

---
# Part 1 — Structural Graph Construction

Constructs the weighted directed graph for the 93-neuron L4-V1 scan-9.3 subnetwork. Edge weight = synaptic cleft area in voxels.

## §1 — Imports

In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import pickle
import os
from pathlib import Path

import microns_datacleaner as mic
import microns_datacleaner.filters as fl

## §2 — Load and filter neurons

In [78]:
cleaner = mic.MicronsDataCleaner(datadir="data", version=1718, download_policy='minimum')
units, _ = cleaner.process_nucleus_data(functional_data='best_only')

proofread = fl.filter_neurons(units, proofread='ax_clean')
matched   = fl.filter_neurons(proofread, tuning='matched').reset_index(drop=True)

print(f"Full working set: {len(matched)} neurons")

Transform positions: 100%|██████████| 94014/94014 [00:00<00:00, 104994.52it/s]


Full working set: 906 neurons


In [79]:
# Session 9.3 V1 only: session=9, scan_idx=3, brain_area='V1'
sess93_v1 = matched[
    (matched['session']    == 9) &
    (matched['scan_idx']   == 3) &
    (matched['brain_area'] == 'V1')
].reset_index(drop=True)

print(f"Session 9.3 V1 neurons: {len(sess93_v1)}")
print(f"Layer breakdown:     {sess93_v1['layer'].value_counts().to_dict()}")
print(f"Cell type breakdown: {sess93_v1['cell_type'].value_counts().to_dict()}")

Session 9.3 V1 neurons: 93
Layer breakdown:     {'L4': 93}
Cell type breakdown: {'4P': 93}


## §3 — Load synapse table

In [80]:
SYN_CSV = "data/1718/raw/synapses_matched.csv"

# Merge raw batch files only if the merged CSV does not exist yet.
# (Re-merging fails on some systems due to encoding issues in the batch files.)
if not Path(SYN_CSV).exists():
    import glob
    batch_files = sorted(glob.glob("data/1718/raw/synapses/*.csv"))
    parts = [pd.read_csv(f, encoding='latin-1') for f in batch_files]
    pd.concat(parts, ignore_index=True).to_csv(SYN_CSV, index=False)
    print(f"Merged {len(batch_files)} batch files → {SYN_CSV}")
else:
    print(f"Using existing {SYN_CSV}")

synapses = pd.read_csv(SYN_CSV)
synapses = synapses[synapses['pre_pt_root_id'] != synapses['post_pt_root_id']].reset_index(drop=True)

# CRITICAL: CSV stores 18-digit neuron IDs as float64 — cast to int64
# so they match node keys in NetworkX exactly (no precision loss).
synapses['pre_pt_root_id']  = synapses['pre_pt_root_id'].astype('int64')
synapses['post_pt_root_id'] = synapses['post_pt_root_id'].astype('int64')
matched['pt_root_id']       = matched['pt_root_id'].astype('int64')

print(f"pre dtype: {synapses['pre_pt_root_id'].dtype}")  # must say int64
print(f"Full synapse table:   {len(synapses):,} connections")
print(f"Size range:           {synapses['size'].min():.0f} – {synapses['size'].max():.0f} voxels")
print(f"Mean size:            {synapses['size'].mean():.0f} voxels")
print()

ids_93 = set(sess93_v1['pt_root_id'].astype('int64').tolist())
syn_93 = synapses[
    synapses['pre_pt_root_id'].isin(ids_93) &
    synapses['post_pt_root_id'].isin(ids_93)
].reset_index(drop=True)

print(f"Session 9.3 V1 table: {len(syn_93):,} connections")
if len(syn_93) > 0:
    print(f"Size range:           {syn_93['size'].min():.0f} – {syn_93['size'].max():.0f} voxels")
    print(f"Mean size:            {syn_93['size'].mean():.0f} voxels")

Using existing data/1718/raw/synapses_matched.csv
pre dtype: int64
Full synapse table:   182,352 connections
Size range:           160 – 424532 voxels
Mean size:            9324 voxels

Session 9.3 V1 table: 229 connections
Size range:           260 – 57268 voxels
Mean size:            7542 voxels


## §4 — Build weighted directed graph

In [81]:
def build_weighted_digraph(neurons_df, syn_df):
    G = nx.DiGraph()

    # Use .tolist() to convert to Python ints — avoids the iterrows() bug where
    # int64 columns get silently recast to float64 in a mixed-dtype Series,
    # which loses precision on 18-digit IDs and breaks NetworkX's hash lookup.
    node_ids    = neurons_df['pt_root_id'].astype('int64').tolist()
    layers      = neurons_df['layer'].tolist()
    cell_types  = neurons_df['cell_type'].tolist()
    brain_areas = neurons_df['brain_area'].tolist()
    pref_oris   = neurons_df['pref_ori'].tolist()
    gOSIs       = neurons_df['gOSI'].tolist()
    xs          = neurons_df['pt_position_x'].tolist()
    ys          = neurons_df['pt_position_y'].tolist()
    zs          = neurons_df['pt_position_z'].tolist()

    for i, nid in enumerate(node_ids):
        G.add_node(nid, layer=layers[i], cell_type=cell_types[i],
                   brain_area=brain_areas[i], pref_ori=pref_oris[i],
                   gOSI=gOSIs[i], x=xs[i], y=ys[i], z=zs[i])

    node_set = set(G.nodes())
    pre_ids  = syn_df['pre_pt_root_id'].astype('int64').tolist()
    post_ids = syn_df['post_pt_root_id'].astype('int64').tolist()
    weights  = syn_df['size'].tolist()

    edges_added = 0
    for pre, post, w in zip(pre_ids, post_ids, weights):
        if pre in node_set and post in node_set:
            G.add_edge(pre, post, weight=w)
            edges_added += 1

    print(f"  Nodes: {G.number_of_nodes()}, edges added: {edges_added}")
    return G

print("Building 906-neuron network...")
G_906 = build_weighted_digraph(matched, synapses)

print("Building 93-neuron network...")
G_93  = build_weighted_digraph(sess93_v1, syn_93)

print()
print(f"Full 906-neuron network:     {G_906.number_of_nodes()} nodes, {G_906.number_of_edges():,} edges, density={nx.density(G_906):.4f}")
print(f"Session 9.3 V1 network (93): {G_93.number_of_nodes()}  nodes, {G_93.number_of_edges():,} edges,  density={nx.density(G_93):.4f}")


Building 906-neuron network...
  Nodes: 906, edges added: 11822
Building 93-neuron network...
  Nodes: 93, edges added: 229

Full 906-neuron network:     906 nodes, 11,822 edges, density=0.0144
Session 9.3 V1 network (93): 93  nodes, 229 edges,  density=0.0268


## §5 — Save graphs

In [82]:
# Save both graphs for use in other notebooks
os.makedirs('data', exist_ok=True)

with open('data/G_906.pkl', 'wb') as f:
    pickle.dump(G_906, f)
with open('data/G_93.pkl', 'wb') as f:
    pickle.dump(G_93, f)

print("Saved: data/G_906.pkl  and  data/G_93.pkl")
print("Reload with:  G = pickle.load(open('data/G_906.pkl', 'rb'))")

Saved: data/G_906.pkl  and  data/G_93.pkl
Reload with:  G = pickle.load(open('data/G_906.pkl', 'rb'))


---
# Part 2 — Export Node and Edge Tables

Loads the graph pickle from Part 1 and exports `G_93_nodes.csv` and `G_93_edges.csv` to `data/exports/`.

## §1 — Imports

In [83]:
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os


## §2 — Load graph

In [84]:
with open('data/G_906.pkl', 'rb') as f:
    G_906 = pickle.load(f)
with open('data/G_93.pkl', 'rb') as f:
    G_93 = pickle.load(f)

print(f'G_906: {G_906.number_of_nodes()} nodes, {G_906.number_of_edges():,} edges')
print(f'G_93:  {G_93.number_of_nodes()} nodes,  {G_93.number_of_edges():,} edges')


G_906: 906 nodes, 11,822 edges
G_93:  93 nodes,  229 edges


## §3 — Export node and edge tables

In [85]:
os.makedirs('data/exports', exist_ok=True)

def export_graph(G, prefix):
    rows = []
    for n in G.nodes():
        d = G.nodes[n]
        rows.append({
            'neuron_id':    n,
            'brain_area':   d.get('brain_area'),
            'layer':        d.get('layer'),
            'cell_type':    d.get('cell_type'),
            'pref_ori':     d.get('pref_ori'),
            'gOSI':         d.get('gOSI'),
            'x':            d.get('x'),
            'y':            d.get('y'),
            'z':            d.get('z'),
            'out_degree':   G.out_degree(n),
            'in_degree':    G.in_degree(n),
            'out_strength': G.out_degree(n, weight='weight'),
            'in_strength':  G.in_degree(n, weight='weight'),
        })
    nodes_df = pd.DataFrame(rows)
    nodes_df.to_csv(f'data/exports/{prefix}_nodes.csv', index=False)
    print(f'Saved data/exports/{prefix}_nodes.csv  ({len(nodes_df)} rows)')

    edges = [(u, v, G[u][v]['weight']) for u, v in G.edges()]
    edges_df = pd.DataFrame(edges,
                             columns=['pre_neuron_id', 'post_neuron_id', 'synapse_size'])
    edges_df.to_csv(f'data/exports/{prefix}_edges.csv', index=False)
    print(f'Saved data/exports/{prefix}_edges.csv  ({len(edges_df)} rows)')

export_graph(G_906, 'G_906')
export_graph(G_93,  'G_93')


Saved data/exports/G_906_nodes.csv  (906 rows)
Saved data/exports/G_906_edges.csv  (11822 rows)
Saved data/exports/G_93_nodes.csv  (93 rows)
Saved data/exports/G_93_edges.csv  (229 rows)


---
# Part 3 — Functional Correlation Matrix

Extracts deconvolved calcium traces for the 93-neuron V1 cohort from the MICrONS functional HDF5 and computes the 93×93 Pearson trace-correlation matrix.

## §1 — Imports

In [86]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import microns_datacleaner as mic
import microns_datacleaner.filters as fl

## §2 — Load cohort

In [87]:
cleaner = mic.MicronsDataCleaner(datadir="data/", version=1718, download_policy='minimum')
units, segments = cleaner.process_nucleus_data(functional_data='best_only')

matched = units[units['tuning_type'] == 'matched']
proofread_matched = matched[matched['strategy_axon'] != 'none']

print(f"Total cells: {len(units)}")
print(f"Functionally matched: {len(matched)}")
print(f"Matched + proofread: {len(proofread_matched)}")

Transform positions: 100%|██████████| 94014/94014 [00:00<00:00, 105160.03it/s]


Total cells: 90434
Functionally matched: 13005
Matched + proofread: 906


## §3 — Filter to scan 9.3 V1

In [88]:
SESSION = 9
SCAN = 3

# Filter to one specific scan
cohort = proofread_matched[
    (proofread_matched['session'] == SESSION) & 
    (proofread_matched['scan_idx'] == SCAN)
].reset_index(drop=True)

# Add a stable index for matching with the structural matrix later
cohort['matrix_index'] = np.arange(len(cohort))

print(f"Cohort: {len(cohort)} neurons in session {SESSION}, scan {SCAN}")

# unit_id is the index used to look up the neuron's trace in microns.h5
unit_ids = cohort['unit_id'].astype(int).values
print(f"unit_ids shape: {unit_ids.shape}")
print(f"unit_ids range: {unit_ids.min()} to {unit_ids.max()}")

Cohort: 99 neurons in session 9, scan 3
unit_ids shape: (99,)
unit_ids range: 1521 to 8608


## §4 — Connect to functional HDF5

In [89]:
funcreader = mic.MicronsFunctionalReader(path="data/functional/microns_functional.h5")
print("Connected to functional data")

Connected to functional data


## §5 — Read trial responses

In [90]:
sesh_scan = f"{SESSION}_{SCAN}"

trial_responses = []
trial = 0

while True:
    try:
        r = funcreader.get_trial(sesh_scan, trial, "V1")
        trial_responses.append(r['responses'])
        trial += 1
    except Exception as e:
        print(f"Stopped at trial {trial}: {type(e).__name__}")
        break

print(f"Found {trial} trials in scan {sesh_scan}")
if trial_responses:
    print(f"First trial shape (all V1 neurons × frames): {trial_responses[0].shape}")

Stopped at trial 464: ValueError
Found 464 trials in scan 9_3
First trial shape (all V1 neurons × frames): (5838, 75)


## §6 — Map cohort unit IDs to HDF5 row indices

In [91]:
with h5py.File("data/functional/microns_functional.h5", "r") as f:
    all_unit_ids = f[f"sessions/{sesh_scan}/meta/unit_ids"][:]
    v1_indices = f[f"sessions/{sesh_scan}/meta/area_indices/V1"][:]

# Unit_ids of the V1 neurons in the order get_trial returns them
v1_unit_ids = all_unit_ids[v1_indices]
print(f"V1 unit_ids in scan {sesh_scan}: {len(v1_unit_ids)} neurons")

# Build a lookup: unit_id -> row index in the V1 response matrix
unit_id_to_row = {int(uid): i for i, uid in enumerate(v1_unit_ids)}

# Find row indices for our cohort
cohort_unit_ids = cohort['unit_id'].astype(int).values
missing = [uid for uid in cohort_unit_ids if uid not in unit_id_to_row]
print(f"Cohort unit_ids missing from V1 list: {len(missing)}")

cohort_rows = np.array([unit_id_to_row[uid] for uid in cohort_unit_ids if uid in unit_id_to_row])
print(f"Cohort rows: shape {cohort_rows.shape}, range {cohort_rows.min()} to {cohort_rows.max()}")

V1 unit_ids in scan 9_3: 5838 neurons
Cohort unit_ids missing from V1 list: 6
Cohort rows: shape (93,), range 1180 to 5817


In [92]:
# Keep only cohort neurons that are present in the V1 response matrix
cohort = cohort[cohort['unit_id'].isin(unit_id_to_row)].reset_index(drop=True)
cohort['matrix_index'] = np.arange(len(cohort))

# Recompute the row indices in the V1 response matrix
cohort_rows = np.array([unit_id_to_row[int(uid)] for uid in cohort['unit_id']])

print(f"Final cohort: {len(cohort)} neurons")
print(f"cohort_rows shape: {cohort_rows.shape}")

Final cohort: 93 neurons
cohort_rows shape: (93,)


## §7 — Concatenate traces

In [93]:
# Extract our cohort's traces from each trial and concatenate
my_traces = [resp[cohort_rows, :] for resp in trial_responses]
traces = np.concatenate(my_traces, axis=1)

print(f"Traces shape: {traces.shape}")
print(f"  ({traces.shape[0]} neurons × {traces.shape[1]} time frames)")

Traces shape: (93, 37840)
  (93 neurons × 37840 time frames)


## §8 — Compute trace-correlation matrix

In [94]:
F = np.corrcoef(traces)

print(f"Functional matrix shape: {F.shape}")
print(f"Diagonal (should all be 1.0): {F.diagonal()[:5]}")
print(f"Symmetric? F[5,10]={F[5,10]:.4f}, F[10,5]={F[10,5]:.4f}")
print(f"Any NaNs? {np.isnan(F).any()}")

Functional matrix shape: (93, 93)
Diagonal (should all be 1.0): [1. 1. 1. 1. 1.]
Symmetric? F[5,10]=-0.0021, F[10,5]=-0.0021
Any NaNs? False


## §9 — Save outputs

In [95]:
import os
import json

out_dir = 'outputs/functional_network'
os.makedirs(out_dir, exist_ok=True)

# Save the correlation matrix
np.save(f'{out_dir}/F_correlation_matrix.npy', F)

# Save the cohort with matrix indices so other notebooks can align
cohort_to_save = cohort[[
    'matrix_index', 'nucleus_id', 'pt_root_id', 'classification_system',
    'cell_type', 'layer', 'brain_area', 'strategy_axon', 'strategy_dendrite',
    'pt_position_x', 'pt_position_y', 'pt_position_z',
    'session', 'scan_idx', 'unit_id', 'cc_abs'
]].copy()
cohort_to_save.to_csv(f'{out_dir}/functional_cohort.csv', index=False)

# Upper triangle (off-diagonal pairs) for summary stats
upper = F[np.triu_indices_from(F, k=1)]

# Save metadata about how the network was built
metadata = {
    'session': int(SESSION),
    'scan': int(SCAN),
    'n_neurons': int(len(cohort)),
    'n_trials': int(trial),
    'n_time_frames': int(traces.shape[1]),
    'correlation_method': 'pearson',
    'data_source': 'concatenated trials, V1 only',
    'mean_correlation': float(upper.mean()),
    'median_correlation': float(np.median(upper)),
    'pairs_above_0.2': int((upper > 0.2).sum()),
    'pairs_above_0.3': int((upper > 0.3).sum()),
}
with open(f'{out_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {out_dir}/")
print(f"  F_correlation_matrix.npy: {F.shape}")
print(f"  functional_cohort.csv: {len(cohort)} neurons")
print(f"  metadata.json")

Saved to outputs/functional_network/
  F_correlation_matrix.npy: (93, 93)
  functional_cohort.csv: 93 neurons
  metadata.json
